In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train_file = '/content/data/fingerprints/android_session_1_1.csv'
test_file = '/content/data/fingerprints/android_session_2_2.csv'

# Carica i dati
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print("Dati caricati con successo!")
print(f"Dimensioni Training Set: {train_df.shape}")
print(f"Dimensioni Test Set: {test_df.shape}\n")

# Prepara i dati per Scikit-learn
X_train = train_df.drop(columns=['room', 'artwork'])
y_train_room = train_df['room']
y_train_artwork = train_df['artwork']

X_test = test_df.drop(columns=['room', 'artwork'])
y_test_room = test_df['room']
y_test_artwork = test_df['artwork']

artwork_encoder = LabelEncoder()
y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)

print("Dati pronti per l'addestramento.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("Avvio addestramento di base della Regressione Logistica...")

# Scalatura dei Dati
# La Regressione Logistica funziona molto meglio con dati scalati (media=0, varianza=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dati scalati con successo.")

# Crea il modello
# multi_class='multinomial' è l'approccio corretto per la classificazione multi-classe
# solver='lbfgs' è un buon solutore per questo tipo di problema
# max_iter=1000 per dare al modello più tempo per convergere
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', random_state=42, n_jobs=-1, max_iter=1000)

# Addestra il modello per predire l'OPERA D'ARTE
model.fit(X_train_scaled, y_train_artwork_encoded)

# Fai le previsioni sul test set scalato
predictions = model.predict(X_test_scaled)

# Calcola l'accuratezza
accuracy = accuracy_score(y_test_artwork_encoded, predictions)

print("\n--- Risultati del Modello di Base ---")
print(f"Artwork Top-1 Accuracy (BAR 1,2): {accuracy * 100:.2f}%")

# Confronto con la baseline
print(f"Baseline 57-NN (dal paper): 85.70%")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import pandas as pd
import os

print("Avvio addestramento e salvataggio della Regressione Logistica...")

# Scalatura Dati
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Impostazioni del modello
# Usiamo gli stessi parametri per entrambi i modelli
lr_params = {
    'multi_class': 'multinomial',
    'solver': 'lbfgs',
    'random_state': 42,
    'n_jobs': -1,
    'max_iter': 1000
}

# Addestramento e Previsione STANZA
lr_room = LogisticRegression(**lr_params)
lr_room.fit(X_train_scaled, y_train_room)
room_predictions = lr_room.predict(X_test_scaled)
room_accuracy = accuracy_score(y_test_room, room_predictions)

# Addestramento e Previsione OPERA D'ARTE
lr_artwork = LogisticRegression(**lr_params)
lr_artwork.fit(X_train_scaled, y_train_artwork_encoded)
artwork_predictions_encoded = lr_artwork.predict(X_test_scaled)
artwork_accuracy = accuracy_score(y_test_artwork_encoded, artwork_predictions_encoded)

# Calcolo Top-3 Accuracy
artwork_probabilities = lr_artwork.predict_proba(X_test_scaled)
top3_predictions = artwork_probabilities.argsort(axis=1)[:, -3:]
correct_top3 = [y_test_artwork_encoded[i] in top3_predictions[i] for i in range(len(y_test_artwork_encoded))]
top3_accuracy = sum(correct_top3) / len(correct_top3)

# Salvataggio dei Risultati
output_folder = "/content/drive/My Drive/Tesi Magistrale/Risultati_LogisticRegression/"
os.makedirs(output_folder, exist_ok=True)

results_summary = {
    'Dataset': ['BAR 1,2'],
    'Algorithm': ['LogisticRegression_Default'],
    'Room Accuracy': [room_accuracy * 100],
    'Artwork Top-1 Accuracy': [artwork_accuracy * 100],
    'Artwork Top-3 Accuracy': [top3_accuracy * 100]
}
results_df = pd.DataFrame(results_summary)

output_file = os.path.join(output_folder, 'summary_BAR_1_2.csv')
results_df.to_csv(output_file, index=False, float_format='%.2f')

print("\n--- Risultati Finali ---")
print(results_df.to_string(index=False))
print(f"\n Risultati salvati con successo in: {output_file}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import time

print("Avvio dell'ottimizzazione degli iperparametri per la Regressione Logistica...")
start_time = time.time()

# Creiamo una Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(multi_class='multinomial',
                                 solver='lbfgs',
                                 random_state=42,
                                 n_jobs=-1,
                                 max_iter=1000))
])

param_grid = {
    'model__C': [0.1, 1, 10, 100]  # Testiamo diversi livelli di regolarizzazione
}

# Configuriamo GridSearchCV
# cv=3 userà la Cross-Validation a 3 fold
grid_search = GridSearchCV(estimator=pipe,
                           param_grid=param_grid,
                           cv=3,
                           verbose=2,
                           scoring='accuracy')

# Avviamo la ricerca!
# Usiamo i dati NON scalati (X_train). La pipeline si occuperà della scalatura.
grid_search.fit(X_train, y_train_artwork_encoded)

end_time = time.time()
print(f"\nRicerca completata in {((end_time - start_time) / 60):.2f} minuti.")

# Mostriamo i risultati
print("\n--- Risultati dell'Ottimizzazione ---")
print(f"Migliori parametri trovati: {grid_search.best_params_}")
print(f"Miglior punteggio di accuratezza (in cross-validation): {grid_search.best_score_ * 100:.2f}%")

# Valutiamo il modello OTTIMIZZATO sul test set
best_lr_model = grid_search.best_estimator_
predictions = best_lr_model.predict(X_test)
optimized_accuracy = accuracy_score(y_test_artwork_encoded, predictions)

print("\n--- Confronto Performance su Test Set (BAR 1,2) ---")
# 'accuracy' è la variabile calcolata nel Blocco 3
print(f"Accuratezza Modello di Default: {accuracy * 100:.2f}%")
print(f"Accuratezza Modello Ottimizzato: {optimized_accuracy * 100:.2f}%")

In [ ]:
import pandas as pd
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("Avvio della suite di esperimenti completa con la Regressione Logistica (Default C=1)...")

# Definisci i percorsi dei file e le combinazioni
data_dir = '/content/data/fingerprints/'
files = [
    'android_session_1_1.csv',
    'android_session_2_2.csv',
    'ios_session_1_3.csv',
    'ios_session_2_4.csv'
]

# Creiamo tutte le 12 combinazioni di train/test
experiments = []
for train_file in files:
    for test_file in files:
        if train_file != test_file:
            experiments.append((train_file, test_file))

# Lista per salvare tutti i risultati
all_results = []

# Definisci la pipeline con i parametri di default (C=1)
pipe_room = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(multi_class='multinomial',
                                 solver='lbfgs',
                                 random_state=42,
                                 n_jobs=-1,
                                 max_iter=1000))
])

pipe_artwork = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(multi_class='multinomial',
                                 solver='lbfgs',
                                 random_state=42,
                                 n_jobs=-1,
                                 max_iter=1000))
])

# Ciclo su tutti gli esperimenti
for i, (train_file, test_file) in enumerate(experiments):
    print(f"\n--- Esecuzione Esperimento {i+1}/{len(experiments)}: Train={train_file}, Test={test_file} ---")
    start_time = time.time()

    # Carica i dati
    train_df = pd.read_csv(os.path.join(data_dir, train_file))
    test_df = pd.read_csv(os.path.join(data_dir, test_file))

    # Prepara i dati (non scalati, la pipeline li gestirà)
    X_train = train_df.drop(columns=['room', 'artwork'])
    y_train_room = train_df['room']
    y_train_artwork = train_df['artwork']
    X_test = test_df.drop(columns=['room', 'artwork'])
    y_test_room = test_df['room']
    y_test_artwork = test_df['artwork']

    artwork_encoder = LabelEncoder()
    y_train_artwork_encoded = artwork_encoder.fit_transform(y_train_artwork)
    y_test_artwork_encoded = artwork_encoder.transform(y_test_artwork)

    # Addestra e valuta il modello per la STANZA
    pipe_room.fit(X_train, y_train_room)
    room_accuracy = accuracy_score(y_test_room, pipe_room.predict(X_test))

    # Addestra e valuta il modello per l'OPERA D'ARTE
    pipe_artwork.fit(X_train, y_train_artwork_encoded)
    artwork_top1_accuracy = accuracy_score(y_test_artwork_encoded, pipe_artwork.predict(X_test))

    artwork_probabilities = pipe_artwork.predict_proba(X_test)
    top3_predictions = artwork_probabilities.argsort(axis=1)[:, -3:]
    correct_top3 = [y_test_artwork_encoded[j] in top3_predictions[j] for j in range(len(y_test_artwork_encoded))]
    artwork_top3_accuracy = sum(correct_top3) / len(correct_top3)

    # Salva i risultati di questo esperimento
    dataset_name = f"BAR {train_file.split('_')[-1].split('.')[0]},{test_file.split('_')[-1].split('.')[0]}"
    all_results.append({
        'Dataset': dataset_name,
        'Room Accuracy': room_accuracy * 100,
        'Artwork Top-1 Accuracy': artwork_top1_accuracy * 100,
        'Artwork Top-3 Accuracy': artwork_top3_accuracy * 100
    })

    end_time = time.time()
    print(f"Completato in {end_time - start_time:.2f} secondi.")

results_df = pd.DataFrame(all_results)
output_file = "/content/drive/My Drive/Tesi Magistrale/Risultati_LogisticRegression/LR_default_all_experiments.csv"
results_df.to_csv(output_file, index=False, float_format='%.2f')

print("\n\n--- TABELLA RIASSUNTIVA REGRESSIONE LOGISTICA (Default) ---")
print(results_df.to_string(index=False))
print(f"\n Suite di esperimenti completata! Risultati salvati in: {output_file}")